# BIRD Cross-Database Evaluation Quick Start (BigQuery + SQLite Fallback)

This notebook demonstrates how to set up and run a hybrid cross-database evaluation on the BIRD benchmark.

### Process Flow:
1. **Model Generation**: The model generates SQL targeting Google BigQuery.
2. **Execution**: The generated query executes on BigQuery.
3. **Hybrid Verification**: If the reference/golden query fails on BigQuery (which is common since reference queries are in SQLite format), EvalBench executes it on a local SQLite fallback database to fetch the ground truth rows, then performs a structure-agnostic comparison.
4. **Reporting**: Results are stored directly to Google BigQuery, generating a Looker Studio visualization dashboard link.

#### 1. Clone the EvalBench repository from GitHub

In [ ]:
!git clone https://github.com/GoogleCloudPlatform/evalbench.git

#### 2. Install Dependencies
We install `uv` to manage python packages, sync the virtual environment, and install the `mcp` library package.

In [ ]:
!pip install uv
!cd evalbench && uv sync
!cd evalbench && .venv/bin/pip3 install mcp
!cd evalbench && .venv/bin/python3 -m grpc_tools.protoc --proto_path=evalbench/evalproto --python_out=evalbench/evalproto --pyi_out=evalbench/evalproto --grpc_python_out=evalbench/evalproto --experimental_editions evalbench/evalproto/*.proto

#### 3. Download the BIRD Dataset and Database Connections
This script downloads the natural language prompts and all the SQLite databases required for resolving the ground truth evaluation rows.

In [ ]:
!cd evalbench && bash datasets/bird/download_dataset.sh

#### 4. Setup Evalbench Environment and GCP Credentials
Enter your GCP Project ID and region where the BigQuery datasets will be loaded.

In [ ]:
import os
os.environ['EVAL_GCP_PROJECT_ID'] = '<put-your-project-id-here>'
os.environ['EVAL_GCP_PROJECT_REGION'] = '<gcp-region-here>'

In [ ]:
from google.colab import auth
auth.authenticate_user(project_id=os.environ['EVAL_GCP_PROJECT_ID'])

#### 5. Run the Hybrid Cross-Database Evaluation
This runs the evaluation using our new `example_hybrid_run_config.yaml` configuration.

In [ ]:
!cd evalbench && .venv/bin/python3 evalbench/evalbench.py --experiment_config="datasets/bird/example_hybrid_run_config.yaml"

#### 6. Inspecting Results in BigQuery
Once the run is complete, you can click the Looker Studio URL printed at the end of the logs to view the visual report. 

Alternatively, you can query the results directly in Python from BigQuery:

In [ ]:
from google.cloud import bigquery
import pandas as pd

project_id = os.environ['EVAL_GCP_PROJECT_ID']
client = bigquery.Client(project=project_id)

# Dynamically filters for the most recent evaluation run
query = f"""
SELECT 
    id, 
    database, 
    nl_prompt, 
    generated_sql, 
    golden_sql, 
    generated_result, 
    golden_result 
FROM `{project_id}.evalbench.results` 
WHERE job_id = (
    SELECT job_id 
    FROM `{project_id}.evalbench.results` 
    ORDER BY run_time DESC 
    LIMIT 1
)
"""
df = client.query(query).to_dataframe()
df